# 测试

In [1]:
import set_env

In [2]:
import torch
from torch import fx
from torch.nn import Module
# import tvm.testing
from tvm.relax.frontend.torch import from_fx
from tvm.contrib.msc.core.frontend import translate
from tvm.contrib.msc.core import utils as msc_utils

In [3]:
class Conv2D1(Module):
    def __init__(self):
        super().__init__()
        self.conv = torch.nn.Conv2d(3, 6, 7, bias=True)

    def forward(self, data):
        return self.conv(data)

In [13]:
input_info = [([1, 3, 10, 10], "float32")]
torch_model = Conv2D1()
graph_model = fx.symbolic_trace(torch_model)
with torch.no_grad():
    mod = from_fx(graph_model, input_info)
mod.show()
graph, _ = translate.from_relax(mod)
inspect = graph.inspect()
type(graph), inspect 

(tvm.contrib.msc.core.ir.graph.MSCGraph,
 {'inputs': [{'name': 'inp_0',
    'shape': [1, 3, 10, 10],
    'dtype': 'float32',
    'layout': 'NCHW'}],
  'outputs': [{'name': 'conv2d',
    'shape': [1, 6, 4, 4],
    'dtype': 'float32',
    'layout': 'NCHW'}],
  'nodes': {'total': 2, 'input': 1, 'msc.conv2d_bias': 1}})

In [14]:
type(graph)

tvm.contrib.msc.core.ir.graph.MSCGraph